# 1 · Simulate spots and noise
Before starting, read [README.md](README.md). Run cells in order. Example files are from DECODE’s experiment, not the Nanoimager. Do not interpret a successful run as validated microscopy results. Nothing trains or writes data unless you explicitly enable it.
DECODE simulates emitters using a spline PSF, background, shot noise and camera response. Here we use the bundled **example EMCCD parameters and PSF**. Their values do not describe the lab Nanoimager. We simulate a small number of frames, not a training run.

Edit `decode_walkthrough/config.yaml` for walkthrough settings, then rerun from the first cell. The example model’s full DECODE parameters remain in its own `param_run.yaml`.


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import tifffile
import yaml

# Start Jupyter from the project folder or decode_walkthrough/.
ROOT = Path.cwd().resolve()
if not (ROOT / 'decode_walkthrough' / 'config.yaml').is_file():
    ROOT = ROOT.parent
config_path = ROOT / 'decode_walkthrough' / 'config.yaml'
if not config_path.is_file():
    raise FileNotFoundError('Start Jupyter in the project folder or decode_walkthrough/.')
with config_path.open() as f:
    CFG = yaml.safe_load(f)
def project_path(value):
    if value is None: return None
    p = Path(value).expanduser()
    return p if p.is_absolute() else ROOT / p
EXAMPLE = project_path(CFG['paths']['example_folder'])
WORK = project_path(CFG['paths']['work_folder'])
print('Project:', ROOT, '| Config:', config_path, '| Example exists:', EXAMPLE.exists())


In [ ]:
import torch
import decode
import decode.utils.param_io
from decode.neuralfitter.train.random_simulation import setup_random_simulation
print('PyTorch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())


### Set example simulation parameters
`emitter_av` is the average number of emitters in each 40 × 40 frame. Background and brightness are photons under the model convention. The camera adds noise; `convert2photons=True` converts the simulated camera output back into estimated photon units. Change values and rerun. Keep the example camera parameters unchanged for the example PSF; real sCMOS values must be measured.


In [ ]:
param = decode.utils.param_io.load_params(str(project_path(CFG['paths']['example_parameters'])))
param.InOut.calibration_file = str(project_path(CFG['paths']['example_calibration']))
sim_cfg = CFG['example_simulation']
param.Hardware.device_simulation = sim_cfg['device']
param.Hardware.device = sim_cfg['device']
param.Simulation.emitter_av = sim_cfg['emitter_average']
param.Simulation.bg_uniform = sim_cfg['background_photons']
param.Simulation.intensity_mu_sig = sim_cfg['brightness_mean_std']
param.HyperParameter.pseudo_ds_size = sim_cfg['last_frame_index']  # inclusive
param.TestSet.test_size = sim_cfg['last_frame_index']
torch.manual_seed(sim_cfg['seed'])
np.random.seed(sim_cfg['seed'])  # DECODE uses both NumPy and PyTorch random generators
sim, _ = setup_random_simulation(param)
emitters, frames, backgrounds = sim.sample()
frames = frames.detach().cpu().numpy()
print('Simulated frames:', frames.shape, '| convert to photons:', param.Camera.convert2photons)
plt.figure(figsize=(6,5)); plt.imshow(frames[0], cmap='gray'); plt.colorbar(label='model output'); plt.title('DECODE example simulation'); plt.show()


### Compare both raw channels with the same display scale
Choose your own TIFF; this cell reads only one frame. The example simulation is in photon units, but your raw TIFF is in camera ADU; do not compare numerical values until you have a validated camera conversion. Auto-contrast can make a background look white.


In [ ]:
RAW_TIFF = project_path(CFG['own_data']['raw_tiff'])
FRAME = CFG['comparison']['frame_index']
if RAW_TIFF is None:
    print('Set own_data.raw_tiff in config.yaml to compare both halves.')
else:
    with tifffile.TiffFile(RAW_TIFF) as tif:
        raw = tif.pages[FRAME].asarray()
    if raw.ndim != 2 or raw.shape[1] % 2: raise ValueError('Expected a 2D frame with two equal-width channels.')
    left, right = np.split(raw, 2, axis=1)
    limits = CFG['comparison']
    vmin, vmax = limits['display_min_adu'], limits['display_max_adu']
    if vmin is None and vmax is None:
        vmin, vmax = np.percentile(raw, limits['percentiles'])
    elif vmin is None or vmax is None or vmin >= vmax:
        raise ValueError('Set both valid comparison display limits, or leave both null.')
    fig, axes = plt.subplots(1, 2, figsize=(12,5))
    for ax, img, name in zip(axes, [left, right], ['Left', 'Right']):
        im = ax.imshow(img, cmap='gray', vmin=vmin, vmax=vmax); ax.set_title(name); print(name, 'ADU percentiles:', np.percentile(img, [1,50,99,99.9]))
    fig.colorbar(im, ax=axes, label='ADU'); plt.show()
